# 01 - Bronze Layer: Raw Data Ingestion
Read raw CSV files from Chicago and Dallas and persist them as managed Delta tables.
Also export Parquet snapshots for raw and Bronze layers.

In [0]:
%run ./00_setup_config

# 00 - Setup & Configuration
This notebook sets up schemas, paths, and runtime variables for the Food Inspection project.

Medallion layers are separated by schema for Delta tables and by folder path for Parquet snapshots.

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


All widgets cleared.


## Widget Parameters

raw_path    = /Volumes/workspace/food_inspection/raw_data
bronze_path = /Volumes/workspace/foodinspection_bronze/food_inspection
silver_path = /Volumes/workspace/foodinspection_silver/food_inspection
gold_path   = /Volumes/workspace/foodinspection_gold/food_inspection
bronze_schema = foodinspection_bronze
silver_schema = foodinspection_silver
gold_schema   = foodinspection_gold


In [0]:
print("RAW_CHICAGO_PATH =", RAW_CHICAGO_PATH)

df_test = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .csv(RAW_CHICAGO_PATH)
)

print("Bronze test read rows:", df_test.count())
print("Bronze test read columns:", len(df_test.columns))
df_test.printSchema()


RAW_CHICAGO_PATH = /Volumes/workspace/food_inspection/raw_data/Food_Inspections_20260411.csv
Bronze test read rows: 308431
Bronze test read columns: 17
root
 |-- Inspection ID: integer (nullable = true)
 |-- DBA Name: string (nullable = true)
 |-- AKA Name: string (nullable = true)
 |-- License #: integer (nullable = true)
 |-- Facility Type: string (nullable = true)
 |-- Risk: string (nullable = true)
 |-- Address: string (nullable = true)
 |-- City: string (nullable = true)
 |-- State: string (nullable = true)
 |-- Zip: integer (nullable = true)
 |-- Inspection Date: date (nullable = true)
 |-- Inspection Type: string (nullable = true)
 |-- Results: string (nullable = true)
 |-- Violations: string (nullable = true)
 |-- Latitude: string (nullable = true)
 |-- Longitude: string (nullable = true)
 |-- Location: string (nullable = true)



Raw Path         : /Volumes/workspace/food_inspection/raw_data
Raw Chicago Path : /Volumes/workspace/food_inspection/raw_data/Food_Inspections_20260411.csv
Raw Dallas Path  : /Volumes/workspace/food_inspection/raw_data/Restaurant_and_Food_Establishment_Inspections_(October_2016_to_January_2024)_20260411.csv
Bronze Schema    : foodinspection_bronze
Silver Schema    : foodinspection_silver
Gold Schema      : foodinspection_gold
Bronze Path      : /Volumes/workspace/foodinspection_bronze/food_inspection
Silver Path      : /Volumes/workspace/foodinspection_silver/food_inspection
Gold Path        : /Volumes/workspace/foodinspection_gold/food_inspection
Raw Snapshot Path: /Volumes/workspace/food_inspection/raw_data/parquet_snapshots
Snapshot Run Id  : 20260413_181103
Snapshot Date    : 2026/04/13


## Create Medallion Schemas

Schemas ready: foodinspection_bronze, foodinspection_silver, foodinspection_gold


## Verify Raw Data Files Exist

Raw data files:
  Food_Inspections_20260411.csv (330.70 MB)
  Restaurant_and_Food_Establishment_Inspections_(October_2016_to_January_2024)_20260411.csv (192.79 MB)
  backup_Food_Inspections_20260411.csv (330.70 MB)
  parquet_snapshots/ (0.00 MB)


## Helper Utilities

Configuration ready. Variables and snapshot helpers are available via %run ./00_setup_config


In [0]:
import re
from pyspark.sql.functions import lit, current_timestamp


def sanitize_columns(df):
    """Replace spaces and special characters in column names with underscores."""
    for old_name in df.columns:
        new_name = re.sub(r'[^a-zA-Z0-9_]', '_', old_name.strip()).strip('_')
        new_name = re.sub(r'_+', '_', new_name)
        df = df.withColumnRenamed(old_name, new_name)
    return df


def get_etl_job_id():
    """Get the current notebook run ID, or 'interactive' if running manually."""
    try:
        return dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
    except Exception:
        return "interactive"


ETL_JOB_ID = get_etl_job_id()


def add_bronze_lineage(df, source_name):
    """Add lineage/audit columns to Bronze layer DataFrame."""
    return (
        df
        .withColumn("data_source_name", lit(source_name))
        .withColumn("ingest_timestamp", current_timestamp())
        .withColumn("etl_job_id", lit(ETL_JOB_ID))
    )

## 1. Ingest Chicago Food Inspections

In [0]:
df_chicago_raw = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(RAW_CHICAGO_PATH)
)

print(f"Chicago raw record count: {df_chicago_raw.count()}")
print(f"Chicago raw column count: {len(df_chicago_raw.columns)}")
df_chicago_raw.printSchema()
display(df_chicago_raw.limit(10))

# Raw-zone Parquet snapshot
write_parquet_snapshot(df_chicago_raw, RAW_SNAPSHOT_PATH, "chicago_inspections")

# Bronze Delta table + Bronze snapshot

df_chicago_raw = sanitize_columns(df_chicago_raw)
df_chicago_raw = add_bronze_lineage(df_chicago_raw, "Chicago_OpenData_FoodInspections")

(
    df_chicago_raw.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", True)
    .saveAsTable(f"{BRONZE_SCHEMA}.chicago_inspections")
)
write_parquet_snapshot(df_chicago_raw, BRONZE_PATH, "chicago_inspections")
print("Bronze table bronze.chicago_inspections created successfully.")

Chicago raw record count: 308431
Chicago raw column count: 17
root
 |-- Inspection ID: integer (nullable = true)
 |-- DBA Name: string (nullable = true)
 |-- AKA Name: string (nullable = true)
 |-- License #: integer (nullable = true)
 |-- Facility Type: string (nullable = true)
 |-- Risk: string (nullable = true)
 |-- Address: string (nullable = true)
 |-- City: string (nullable = true)
 |-- State: string (nullable = true)
 |-- Zip: integer (nullable = true)
 |-- Inspection Date: date (nullable = true)
 |-- Inspection Type: string (nullable = true)
 |-- Results: string (nullable = true)
 |-- Violations: string (nullable = true)
 |-- Latitude: string (nullable = true)
 |-- Longitude: string (nullable = true)
 |-- Location: string (nullable = true)



Inspection ID,DBA Name,AKA Name,License #,Facility Type,Risk,Address,City,State,Zip,Inspection Date,Inspection Type,Results,Violations,Latitude,Longitude,Location
2633665,Union Sushi + BBQ + Izakaya,Union Sushi + BBQ + Izakaya,3073928,Restaurant,Risk 1 (High),1538 N CLYBOURN AVE,CHICAGO,IL,60610,2026-04-08,License,Pass,"10. ADEQUATE HANDWASHING SINKS PROPERLY SUPPLIED AND ACCESSIBLE - Comments: OBSERVED HAND WASHING ONLY SIGNAGE NEED FOR THE ONE COMPARTMENT SINK THAT THEY WILL USE AS A HAND WASHING SINK FOR THE REAR PREP AREA JUST TO THE RIGHT OF THE WALK IN COOLING UNIT. INSTRUCTED TO OBTAIN AND PLACE AT THE SINK UNIT. | 38. INSECTS, RODENTS, & ANIMALS NOT PRESENT - Comments: OBSERVED A GAP AT THE BOTTOM LEFT OF THE SINGLE EXIT DOOR OF ABOUT 1/4 INCH GAP. INSTRUCTED TO REPLACE OR ADJUST AS A MEANS OF PREVENTING POSSIBLE PEST AND OR INSECT ACTIVITY. | 51. PLUMBING INSTALLED; PROPER BACKFLOW DEVICES - Comments: OBSERVED A LEAK AT THE RIGHT FAUCET OF THE FOUR COMPARTMENT SINK LOCATED AT THE BAR. INSTRUCTED TO REPAIR AND MAINTAIN IN GOOD WORKING ORDER.",41.90931377383,-87.64750167889,"(41.90931377383362, -87.64750167889011)"
2633654,TAMALES EXPRESS,TAMALES EXPRESS,2873326,Restaurant,Risk 1 (High),3802 W DIVERSEY AVE,CHICAGO,IL,60647,2026-04-08,Canvass,Pass,"36. THERMOMETERS PROVIDED & ACCURATE - Comments: OBSERVED BROKEN THERMOMETERS INSIDE UPRIGHT REFRIGERATION UNITS IN FRONT PREP AREA. INSTRUCTED TO REPLACE AND MAINTAIN. | 47. FOOD & NON-FOOD CONTACT SURFACES CLEANABLE, PROPERLY DESIGNED, CONSTRUCTED & USED - Comments: OBSERVED RUSTED SHELVING RACKS INSIDE UPRIGHT COCA COLA COOLING EQUIPMENT. INSTRUCTED TO RESURFACE OR REPLACE TO MAINTAIN IN GOOD REPAIR. | 53. TOILET FACILITIES: PROPERLY CONSTRUCTED, SUPPLIED, & CLEANED - Comments: OBSERVED GARBAGE RECEPTACLE IN WOMENS TOILETROOM WITHOUT COVERING. INSTRUCTED TO PROVIDE AND MAINTAIN. | 55. PHYSICAL FACILITIES INSTALLED, MAINTAINED & CLEAN - Comments: OBSERVED FLOORS IN REAR PREP AREA AROUND ALL HEAVY COOKING EQUIPMENT AND UNDER 3 COMPARTMENT SINK IN NEED OF CLEANING. INSTRUCTED TO PROVIDE AND MAINTAIN FLOOR. | 56. ADEQUATE VENTILATION & LIGHTING; DESIGNATED AREAS USED - Comments: OBSERVED INADEQUATE LIGHTING IN REAR PREP AREA. INSTRUCTED TO REPLACE BURNED OUT BULBS AND MAINTAIN",41.93195943809,-87.72223430062,"(41.931959438085926, -87.72223430061652)"
2633653,MICHAEL FARADAY ELEMENTARY,MICHAEL FARADAY ELEMENTARY,24371,School,Risk 1 (High),3250 W Monroe St (100S),CHICAGO,IL,60624,2026-04-08,Canvass Re-Inspection,Pass,"47. FOOD & NON-FOOD CONTACT SURFACES CLEANABLE, PROPERLY DESIGNED, CONSTRUCTED & USED - Comments: 4-501.11 OBSERVED DAMAGED GASKET ON REACH-IN COOLER DOOR NEAR SERVING AREA. INSTRUCTED MANAGER TO REPLACE COOLER DOOR GASKET AND MAINTAIN.",41.87974117351,-87.70808198851,"(41.879741173507504, -87.70808198850602)"
2633641,PENNY'S ROSCOE,PENNY'S NOODLE SHOP,2677553,Restaurant,Risk 1 (High),3400 N SHEFFIELD AVE,CHICAGO,IL,60657,2026-04-08,Canvass Re-Inspection,Pass,null,41.94361136465,-87.6543689053,"(41.9436113646505, -87.65436890530064)"
2633683,POP UP BAGELS,POP UP BAGELS,3078489,Restaurant,Risk 2 (Medium),2321 N LINCOLN AVE,CHICAGO,IL,60614,2026-04-08,License,Pass,null,41.92420792024,-87.64681810947,"(41.924207920242395, -87.6468181094676)"
2633675,Union Sushi + BBQ + Izakaya,Union Sushi + BBQ + Izakaya,3073929,Restaurant,Risk 3 (Low),1538 N CLYBOURN AVE,CHICAGO,IL,60610,2026-04-08,License,Pass,null,41.90931377383,-87.64750167889,"(41.90931377383362, -87.64750167889011)"
2633638,ALI'S FOOD MART,ALI'S FOOD MART,2972505,Grocery Store,Risk 2 (Medium),1308 S CENTRAL PARK AVE,CHICAGO,IL,60623,2026-04-08,Short Form Complaint,Fail,"38. INSECTS, RODENTS, & ANIMALS NOT PRESENT - Comments: OBSERVED OVER 20 RAT DROPPINGS SCATTERED ON FLOOR UNDER DISPLAY SHELVES, NEAR WATER HEATER AND BETWEEN SHELVING IN SALES, PREP AND STORAGE AREAS. INSTRUCTED MANAGER TO CALL AN EXTERMINATOR FOR SERVICE CLEAN AND SANITIZE ALL AREAS. PRIORITY FOUNDATION 7-38-020(A) CITATION ISSUED. | 51. PLUMBIN

Parquet snapshot written to: /Volumes/workspace/food_inspection/raw_data/parquet_snapshots/chicago_inspections/2026/04/13/run_id=20260413_181103
Parquet snapshot written to: /Volumes/workspace/foodinspection_bronze/food_inspection/chicago_inspections/2026/04/13/run_id=20260413_181103
Bronze table bronze.chicago_inspections created successfully.


## 2. Ingest Dallas Food Inspections

In [0]:
df_dallas_raw = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .option("multiLine", True)
    .option("escape", '"')
    .csv(RAW_DALLAS_PATH)
)

print(f"Dallas raw record count: {df_dallas_raw.count()}")
print(f"Dallas raw column count: {len(df_dallas_raw.columns)}")
df_dallas_raw.printSchema()
display(df_dallas_raw.limit(10))

# Raw-zone Parquet snapshot
write_parquet_snapshot(df_dallas_raw, RAW_SNAPSHOT_PATH, "dallas_inspections")

# Bronze Delta table + Bronze snapshot

df_dallas_raw = sanitize_columns(df_dallas_raw)
df_dallas_raw = add_bronze_lineage(df_dallas_raw, "Dallas_OpenData_FoodInspections")

(
    df_dallas_raw.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", True)
    .saveAsTable(f"{BRONZE_SCHEMA}.dallas_inspections")
)
write_parquet_snapshot(df_dallas_raw, BRONZE_PATH, "dallas_inspections")
print("Bronze table bronze.dallas_inspections created successfully.")

Dallas raw record count: 78984
Dallas raw column count: 114
root
 |-- Restaurant Name: string (nullable = true)
 |-- Inspection Type: string (nullable = true)
 |-- Inspection Date: date (nullable = true)
 |-- Inspection Score: integer (nullable = true)
 |-- Street Number: integer (nullable = true)
 |-- Street Name: string (nullable = true)
 |-- Street Direction: string (nullable = true)
 |-- Street Type: string (nullable = true)
 |-- Street Unit: string (nullable = true)
 |-- Street Address: string (nullable = true)
 |-- Zip Code: string (nullable = true)
 |-- Violation Description - 1: string (nullable = true)
 |-- Violation Points - 1: integer (nullable = true)
 |-- Violation Detail - 1: string (nullable = true)
 |-- Violation Memo - 1: string (nullable = true)
 |-- Violation Description - 2: string (nullable = true)
 |-- Violation Points - 2: integer (nullable = true)
 |-- Violation Detail - 2: string (nullable = true)
 |-- Violation Memo - 2: string (nullable = true)
 |-- Violation

Restaurant Name,Inspection Type,Inspection Date,Inspection Score,Street Number,Street Name,Street Direction,Street Type,Street Unit,Street Address,Zip Code,Violation Description - 1,Violation Points - 1,Violation Detail - 1,Violation Memo - 1,Violation Description - 2,Violation Points - 2,Violation Detail - 2,Violation Memo - 2,Violation Description - 3,Violation Points - 3,Violation Detail - 3,Violation Memo - 3,Violation Description - 4,Violation Points - 4,Violation Detail - 4,Violation Memo - 4,Violation Description - 5,Violation Points - 5,Violation Detail - 5,Violation Memo - 5,Violation Description - 6,Violation Points - 6,Violation Detail - 6,Violation Memo - 6,Violation Description - 7,Violation Points - 7,Violation Detail - 7,Violation Memo - 7,Violation Description - 8,Violation Points - 8,Violation Detail - 8,Violation Memo - 8,Violation Description - 9,Violation Points - 9,Violation Detail - 9,Violation Memo - 9,Violation Description - 10,Violation Points - 10,Violation Detail - 10,Violation Memo - 10,Violation Description - 11,Violation Points - 11,Violation Detail - 11,Violation Memo - 11,Violation Description - 12,Violation Points - 12,Violation Detail - 12,Violation Memo - 12,Violation Description - 13,Violation Points - 13,Violation Detail - 13,Violation Memo - 13,Violation Description - 14,Violation Points - 14,Violation Detail - 14,Violation Memo - 14,Violation Description - 15,Violation Points - 15,Violation Detail - 15,Violation Memo - 15,Violation Description - 16,Violation Points - 16,Violation Detail - 16,Violation Memo - 16,Violation Description - 17,Violation Points - 17,Violation Detail - 17,Violation Memo - 17,Violation Description - 18,Violation Points - 18,Violation Detail - 18,Violation Memo - 18,Violation Description - 19,Violation Points - 19,Violation Detail - 19,Violation Memo - 19,Violation Description - 20,Violation Points - 20,Violation Detail - 20,Violation Memo - 20,Violation Description - 21,Violation Points - 21,Violation Detail - 21,Violation Memo - 21,Violation Description - 22,Violation Points - 22,Violation Detail - 22,Violation Memo - 22,Violation Description - 23,Violation Points - 23,Violation Detail - 23,Violation Memo - 23,Violation Description - 24,Violation Points - 24,Violation Detail - 24,Violation Memo - 24,Violation Description - 25,Violation Points - 25,Violation Detail - 25,Violation Memo - 25,Inspection Month,Inspection Year,Lat Long Location
TIENDA Y RESTAURANTE LA CAMPIñA SALvadorena,Routine,2016-10-03,66,10818,DENNIS,null,RD,102,10818 DENNIS RD 102,75229,"*01 Cooling -- within 2 hours, 135-70øF",3,"228.75 Food. Time and temperature control. (d) Cooling. (1) Cooked temperature/time controlled for safety food shall be cooled: (A) within two hours, from 57 degrees Celsius (135 degrees Fahrenheit) to 21 degrees C (70 degrees Fahrenheit); and",null,"*10 Chlorine sanitizer concentration, minimum temp.",3,"228.111 Equipment, Utensils, and Linens. Equipment, maintenance and operation. (n) Manual and mechanical warewashing equipment, chemical sanitization - temperature, pH, concentration, and hardness. A chemical sanitizer used in a sanitizing solution for a manual or mechanical operation at contact times specified in õ228.118(3) of this title shall meet the criteria in õ228.206(a) of this title, shall be used in accordance with the EPA-approved manufacturer's label use instructions, and shall be used as follows: (1) a chlorine solution shall have a minimum temperature based on the concentration and pH of the solution as listed in the following chart: Figure: 25 TAC õ228.111(n)(1)",null,*31 Handwashing lavatory - accessible,2,"228.149 Water, Plumbing, and Waste. Plumbing, operation and maintenance. (a) Using a handwashing facility. (1) A handwashing facility shall be maintained so that it is accessible at all times for employee use.",null,*21 RFSM - Not On Site,2,"Sec. 17-2.2(c)(1)(D) (c) Registered food service managers. (1) Registered food service managers required. (D) 

Parquet snapshot written to: /Volumes/workspace/food_inspection/raw_data/parquet_snapshots/dallas_inspections/2026/04/13/run_id=20260413_181103
Parquet snapshot written to: /Volumes/workspace/foodinspection_bronze/food_inspection/dallas_inspections/2026/04/13/run_id=20260413_181103
Bronze table bronze.dallas_inspections created successfully.


## 3. Verify Bronze Tables

In [0]:
display(spark.sql(f"SHOW TABLES IN {BRONZE_SCHEMA}"))

display(spark.sql(f"""
    SELECT 'Chicago' AS source, COUNT(*) AS row_count FROM {BRONZE_SCHEMA}.chicago_inspections
    UNION ALL
    SELECT 'Dallas' AS source, COUNT(*) AS row_count FROM {BRONZE_SCHEMA}.dallas_inspections
"""))

display(
    spark.table(f"{BRONZE_SCHEMA}.chicago_inspections")
    .select("data_source_name", "ingest_timestamp", "etl_job_id")
    .limit(5)
)

database,tableName,isTemporary
foodinspection_bronze,chicago_inspections,false
foodinspection_bronze,dallas_inspections,false


source,row_count
Chicago,308431
Dallas,78984


data_source_name,ingest_timestamp,etl_job_id
Chicago_OpenData_FoodInspections,2026-04-13T18:11:19.650Z,/Users/ravi.ara@northeastern.edu/FoodInspection_FinalProject/test/01_bronze_ingestion
Chicago_OpenData_FoodInspections,2026-04-13T18:11:19.650Z,/Users/ravi.ara@northeastern.edu/FoodInspection_FinalProject/test/01_bronze_ingestion
Chicago_OpenData_FoodInspections,2026-04-13T18:11:19.650Z,/Users/ravi.ara@northeastern.edu/FoodInspection_FinalProject/test/01_bronze_ingestion
Chicago_OpenData_FoodInspections,2026-04-13T18:11:19.650Z,/Users/ravi.ara@northeastern.edu/FoodInspection_FinalProject/test/01_bronze_ingestion
Chicago_OpenData_FoodInspections,2026-04-13T18:11:19.650Z,/Users/ravi.ara@northeastern.edu/FoodInspection_FinalProject/test/01_bronze_ingestion
